In [2]:
import torch
import numpy as np
import math

from omnivoice import OmniVoice, VoiceClonePrompt
from omnivoice.utils.audio import load_audio
from omnivoice import OmniVoice, VoiceClonePrompt, OmniVoiceGenerationConfig
from omnivoice.models.omnivoice import _get_time_steps, _gumbel_sample
from IPython.display import Audio, display

### Utils

In [3]:
def prepare_batch_inputs(
    model, task, gen_config
):
    B = task.batch_size
    inputs_list = [
        model._prepare_inference_inputs(
            task.texts[i],
            task.target_lens[i],
            task.ref_texts[i],
            task.ref_audio_tokens[i],
            task.langs[i],
            task.instructs[i],
            gen_config.denoise,
        )
        for i in range(B)
    ]    
    c_lens = [inp["input_ids"].size(2) for inp in inputs_list]
    max_c_len = max(c_lens)
    pad_id = model.config.audio_mask_id  # Or any other tokens

    batch_input_ids = torch.full(
        (2 * B, model.config.num_audio_codebook, max_c_len),
        pad_id,
        dtype=torch.long,
        device=model.device,
    )
    batch_audio_mask = torch.zeros(
        (2 * B, max_c_len), dtype=torch.bool, device=model.device
    )
    batch_attention_mask = torch.zeros(
        (2 * B, 1, max_c_len, max_c_len), dtype=torch.bool, device=model.device
    )

    for i, inp in enumerate(inputs_list):
        c_len, u_len = c_lens[i], task.target_lens[i]

        # Cond (0 ~ B-1)
        batch_input_ids[i, :, :c_len] = inp["input_ids"]
        batch_audio_mask[i, :c_len] = inp["audio_mask"]
        batch_attention_mask[i, :, :c_len, :c_len] = True

        # Uncond (B ~ 2B-1)
        batch_input_ids[B + i, :, :u_len] = inp["input_ids"][..., -u_len:]
        batch_audio_mask[B + i, :u_len] = inp["audio_mask"][..., -u_len:]
        batch_attention_mask[B + i, :, :u_len, :u_len] = True
        if max_c_len > u_len:
            pad_diag = torch.arange(u_len, max_c_len, device=model.device)
            batch_attention_mask[B + i, :, pad_diag, pad_diag] = True

    tokens = torch.full(
        (B, model.config.num_audio_codebook, max(task.target_lens)),
        model.config.audio_mask_id,
        dtype=torch.long,
        device=model.device,
    )

    timesteps = _get_time_steps(
        t_start=0.0,
        t_end=1.0,
        num_step=gen_config.num_step,
        t_shift=gen_config.t_shift,
    ).tolist()
    schedules = []
    for t_len in task.target_lens:
        total_mask = t_len * model.config.num_audio_codebook
        rem = total_mask
        sched = []
        for step in range(gen_config.num_step):
            num = (
                rem
                if step == gen_config.num_step - 1
                else min(
                    math.ceil(total_mask * (timesteps[step + 1] - timesteps[step])),
                    rem,
                )
            )
            sched.append(int(num))
            rem -= int(num)
        schedules.append(sched)

    layer_ids = torch.arange(
        model.config.num_audio_codebook, device=model.device
    ).view(1, -1, 1)   
    return batch_input_ids, batch_audio_mask, batch_attention_mask, schedules, c_lens, layer_ids, tokens

def process_batch_logits(
    # Output
    batch_logits,
    
    # Variable
	B,
 	step,
	batch_input_ids,  
	schedules, 
	c_lens, 
	layer_ids, 
	tokens,
 
	# Model, Config, Task
	model,
 	gen_config,
	task,
):
    with torch.inference_mode():
        for i in range(B):
            k = schedules[i][step]
            if k <= 0:
                continue

            c_len, t_len = c_lens[i], task.target_lens[i]

            # Extract real target Logits
            # [1, C, T, V]
            c_logits = batch_logits[i : i + 1, :, c_len - t_len : c_len, :]
            u_logits = batch_logits[B + i : B + i + 1, :, :t_len, :]

            pred_tokens, scores = model._predict_tokens_with_scoring(
                c_logits, u_logits, gen_config
            )

            scores = scores - (layer_ids * gen_config.layer_penalty_factor)

            if gen_config.position_temperature > 0.0:
                scores = _gumbel_sample(scores, gen_config.position_temperature)

            sample_tokens = tokens[i : i + 1, :, :t_len]
            scores.masked_fill_(
                sample_tokens != model.config.audio_mask_id, -float("inf")
            )

            _, topk_idx = torch.topk(scores.flatten(), k)
            flat_tokens = sample_tokens.flatten()
            flat_tokens[topk_idx] = pred_tokens.flatten()[topk_idx]
            sample_tokens.copy_(flat_tokens.view_as(sample_tokens))

            # Update individual slices into batched structure
            tokens[i : i + 1, :, :t_len] = sample_tokens
            batch_input_ids[i : i + 1, :, c_len - t_len : c_len] = sample_tokens
            batch_input_ids[B + i : B + i + 1, :, :t_len] = sample_tokens 
             
    return batch_input_ids, tokens

### Load Model

In [4]:
model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map="cuda:0",
    dtype=torch.float16,
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/527 [00:00<?, ?it/s]

In [5]:
sum(p.numel() for p in model.parameters())

812921057

In [6]:
model.eval()

OmniVoice(
  (llm): Qwen3Model(
    (embed_tokens): Embedding(151676, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layernorm): Qw

In [7]:
model.config.audio_vocab_size

1025

In [8]:
model.audio_tokenizer.config.frame_rate

25

In [9]:
model.config.num_audio_codebook

8

In [10]:
model.config.audio_mask_id

1024

In [15]:
model.sampling_rate

24000

### Example Input

In [12]:
ref_audio_path = "voice_audio.mp3"
ref_text = 'Terima kasih Anda semua telah memberi kepada saya'
text="Hidup RIDWAN!!!"
saved_path = "batch_voice_audio.pt"

In [17]:
ref_wav = load_audio(ref_audio_path, model.sampling_rate)
ref_rms = float(np.sqrt(np.mean(ref_wav**2)))

In [18]:
ref_wav_tensor = torch.from_numpy(ref_wav).to(model.audio_tokenizer.device)
ref_audio_tokens = model.audio_tokenizer.encode(
    ref_wav_tensor.unsqueeze(0),
).audio_codes.squeeze(0)  # (C, T)

In [19]:
voice_clone_prompt = VoiceClonePrompt(
    ref_audio_tokens=ref_audio_tokens,
    ref_text=ref_text,
    ref_rms=ref_rms,
)

In [27]:
gen_config = OmniVoiceGenerationConfig.from_dict({})

In [28]:
gen_config.position_temperature = 0.0

In [29]:
gen_config

OmniVoiceGenerationConfig(num_step=32, guidance_scale=2.0, t_shift=0.1, layer_penalty_factor=5.0, position_temperature=0.0, class_temperature=0.0, denoise=True, preprocess_prompt=True, postprocess_output=True, audio_chunk_duration=15.0, audio_chunk_threshold=30.0, pad_duration=0.1, fade_duration=0.1)

In [30]:
full_task = model._preprocess_all(
    text=text,
    voice_clone_prompt=voice_clone_prompt,
    preprocess_prompt=gen_config.preprocess_prompt,
)

short_idx, long_idx = full_task.get_indices(
    gen_config, model.audio_tokenizer.config.frame_rate
)
short_task = full_task.slice_task(short_idx)

In [31]:
short_idx, long_idx 

([0], [])

In [33]:
batch_input_ids, batch_audio_mask, batch_attention_mask, schedules, c_lens, layer_ids, tokens = prepare_batch_inputs(model, short_task, gen_config)

In [34]:
B = short_task.batch_size

In [35]:
batch_input_ids.shape, batch_audio_mask.shape, batch_attention_mask.shape

(torch.Size([2, 8, 224]), torch.Size([2, 224]), torch.Size([2, 1, 224, 224]))

In [36]:
save_data = {
    "gen_config": gen_config,
    "short_task": short_task,
    "batch_input_ids": batch_input_ids,
    "batch_audio_mask": batch_audio_mask,
    "batch_attention_mask": batch_attention_mask,
    "schedules": schedules,
    "c_lens": c_lens,
    "layer_ids": layer_ids,
    "tokens": tokens,
}

torch.save(save_data, saved_path)

### Inference

In [37]:
with torch.inference_mode():
    for step in range(gen_config.num_step):
        print(f"Step - {step}")
        
        batch_logits = model(
            input_ids=batch_input_ids,
            audio_mask=batch_audio_mask,
            attention_mask=batch_attention_mask,
        ).logits.to(torch.float32)    
        
        batch_input_ids, tokens = process_batch_logits(
            # Output
            batch_logits,
            
            # Variable
        	B,
         	step,
        	batch_input_ids,  
        	schedules, 
        	c_lens, 
        	layer_ids, 
        	tokens,
         
        	# Model, Config, Task
        	model,
         	gen_config,
        	short_task,
        )

Step - 0
Step - 1
Step - 2
Step - 3
Step - 4
Step - 5
Step - 6
Step - 7
Step - 8
Step - 9
Step - 10
Step - 11
Step - 12
Step - 13
Step - 14
Step - 15
Step - 16
Step - 17
Step - 18
Step - 19
Step - 20
Step - 21
Step - 22
Step - 23
Step - 24
Step - 25
Step - 26
Step - 27
Step - 28
Step - 29
Step - 30
Step - 31


In [38]:
batch_logits.shape

torch.Size([2, 8, 224, 1025])

In [39]:
batch_input_ids.shape, tokens.shape

(torch.Size([2, 8, 224]), torch.Size([1, 8, 47]))

### Post Process

In [40]:
output = [tokens[i, :, : short_task.target_lens[i]] for i in range(B)]

In [41]:
data = output[0].unsqueeze(0).to(model.device)

In [42]:
data.shape

torch.Size([1, 8, 47])

In [43]:
with torch.inference_mode():
    decoded_audio = model.audio_tokenizer.decode(data)

In [44]:
decoded_ref_wav = decoded_audio.audio_values[0][0].cpu().numpy()
display(Audio(decoded_ref_wav, rate=24000))